# 6.6 Support Vector Machines (SVM)

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 6: Classification Algorithms

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-06-classification-algorithms/06.06_support-vector-machines.ipynb)

Companion code for Section 6.6 of the book. Run the cells in order from top to bottom.

**Example 6.6.1**

In [ ]:
from sklearn.svm import SVC, LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import classification_report
import numpy as np
import matplotlib.pyplot as plt

data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                      stratify=y, random_state=42)
scaler = StandardScaler()
X_tr = scaler.fit_transform(X_train)
X_te = scaler.transform(X_test)

# ── Compare kernels ───────────────────────────────────────────────
print(f'{'Kernel':<12} {'C':>6} {'γ':>8} {'Train Acc':>10} {'Test Acc':>10}')
print('='*52)
for kernel in ['linear','rbf','poly','sigmoid']:
    for C in [0.1, 1.0, 10.0]:
        svm = SVC(kernel=kernel, C=C, gamma='scale', random_state=42)
        svm.fit(X_tr, y_train)
        tr = svm.score(X_tr, y_train)
        te = svm.score(X_te, y_test)
        if C == 1.0:  # Print only C=1 for brevity
            print(f'{kernel:<12} {C:>6.1f} {"scale":>8} {tr:>10.4f} {te:>10.4f}')

# ── Grid search for RBF SVM ────────────────────────────────────────
print('\nGrid search for RBF SVM (C, gamma)...')
param_grid = {'C': [0.01, 0.1, 1, 10, 100], 'gamma': [1, 0.1, 0.01, 0.001, 'scale']},
gs = GridSearchCV(SVC(kernel='rbf', probability=True, random_state=42),
                  param_grid[0], cv=5, scoring='accuracy', n_jobs=-1)
gs.fit(X_tr, y_train)
print(f'Best params: {gs.best_params_}')
print(f'Best CV accuracy: {gs.best_score_:.4f}')
print(f'Test accuracy:    {gs.score(X_te, y_test):.4f}')

# ── Support vectors analysis ──────────────────────────────────────
best_svm = gs.best_estimator_
print(f'\nSupport vectors: {best_svm.n_support_} per class')
print(f'Total SVs: {best_svm.n_support_.sum()} out of {len(X_tr)} training samples')
print(f'({best_svm.n_support_.sum()/len(X_tr):.1%} of training data are support vectors)')

# ── Detailed classification report ────────────────────────────────
y_pred_svm = best_svm.predict(X_te)
print('\nSVM Classification Report:')
print(classification_report(y_test, y_pred_svm, target_names=data.target_names))